# `data` — Ingestion & Risk-Exposure Engineering

This notebook turns raw prices into the two things the rest of the pipeline needs:

1. **Market-data matrices** — aligned `[T days x N stocks]` frames of prices, returns and volumes, plus the benchmark (SPY) return series.
2. **Risk exposures** — the right-hand side of the daily neutralization regression: rolling **market beta**, a **log dollar-volume size proxy**, and **sector dummies**.

### The big picture
* **Survivorship warning.** The universe is a static list of *today's* survivors, so realised IC and P&L are structurally optimistic. We shout about this on every load.
* **Beta** is the classic CAPM slope of a stock's returns on the market's, estimated on a rolling 60-day window — computed for all stocks at once, no per-ticker loop.
* **Size** uses `log(price × volume)` as a look-ahead-free stand-in for market cap (we don't have point-in-time shares outstanding).
* **Sectors** become 0/1 dummy columns so the regression can absorb sector-wide moves.

*(Only defines functions — safe to `%run` from `main.ipynb`; the actual download happens there.)*

In [ ]:
%run config.ipynb

In [ ]:
"""Data ingestion and risk-exposure engineering (vectorized, cross-sectional)."""
import warnings
import yfinance as yf
import pandas as pd
import numpy as np
from typing import Dict, Tuple


def load_and_clean_data() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series]:
    """Download prices/volumes and return aligned price, return, volume and SPY-return frames."""
    # Fire the survivorship warning before anything else. It is the single biggest caveat in
    # the whole project, so nobody should be able to run the pipeline without reading it.
    warnings.warn(
        "\n" + "=" * 80 + "\n"
        "SURVIVORSHIP-BIAS WARNING: the universe is a STATIC list of names that survived\n"
        "to today. Delisted / shrunken firms are absent, so historical IC and P&L are\n"
        "structurally inflated. A production book requires a point-in-time universe.\n"
        + "=" * 80 + "\n",
        UserWarning,
    )

    all_tickers = config.TICKERS + [config.BENCHMARK]
    raw = yf.download(all_tickers, start=config.START_DATE, end=config.END_DATE,
                      auto_adjust=True, progress=False)

    # yfinance hands back a MultiIndex on the columns -- (field, ticker), e.g. ('Close','AAPL').
    # Slice out the 'Close' and 'Volume' blocks so we get plain [days x tickers] matrices,
    # which is the shape every cross-sectional operation below expects.
    prices = raw['Close'][config.TICKERS].copy()
    volumes = raw['Volume'][config.TICKERS].copy()
    spy_prices = raw['Close'][config.BENCHMARK].copy()

    # Missing-data policy. Two different problems need two different treatments:
    #  - A name missing most of its history (bad download, or it barely traded) is not
    #    salvageable, so we drop it entirely rather than invent data for it.
    #  - A one-off missing print inside an otherwise healthy series is fine to carry forward,
    #    but we cap it at 5 days so we never manufacture a long flat stretch of fake prices
    #    (a flat price means zero return and zero volatility, which would quietly bias
    #    every statistic we compute later).
    missing_pct = prices.isnull().sum() / len(prices)
    valid = missing_pct[missing_pct < 0.5].index.tolist()
    prices = prices[valid].ffill(limit=5)
    volumes = volumes[valid]

    # Daily simple returns. The first row is all-NaN by construction (no prior price to
    # compare against), so we drop it.
    returns = prices.pct_change().dropna(how='all')
    spy_returns = spy_prices.pct_change().loc[returns.index]  # keep the benchmark on the same dates

    return prices, returns, volumes, spy_returns


def compute_risk_exposures(prices: pd.DataFrame, returns: pd.DataFrame,
                           volumes: pd.DataFrame, spy_returns: pd.Series) -> Dict[str, pd.DataFrame]:
    """Build the three risk exposures (beta, size, sector dummies) we neutralize each signal against."""
    tickers = returns.columns

    # --- 1) Rolling market beta, for every stock at once ---
    # Beta IS a regression slope: it is the coefficient you would get from regressing a stock's
    # returns on the market's returns. For a simple regression (one regressor) that slope has a
    # closed form, so we never need to call a regression routine here --
    #
    #       beta_i = Cov(r_i, r_market) / Var(r_market)
    #
    # -- and pandas can roll both pieces over a window for all 30 stocks simultaneously.
    # Choosing the window length is a bias-variance trade-off: a short window adapts quickly
    # but is noisy (high variance), a long window is stable but goes stale if the stock's true
    # beta drifts (bias). 60 trading days (~3 months) is the conventional compromise.
    lb = config.BETA_LOOKBACK
    cov = returns.rolling(lb).cov(spy_returns)   # each stock's covariance with the market, per day
    spy_var = spy_returns.rolling(lb).var()      # the market's variance on those same days
    beta_df = cov.div(spy_var, axis=0)           # divide each row by that day's market variance

    # The first ~60 rows have no complete window, so beta is NaN there and those days would be
    # dropped from the study. We back-fill them with the first genuine estimate. Be honest about
    # what this costs: it means early dates use a beta computed from slightly later data, a mild
    # look-ahead. It is standard warm-up handling and it only touches the first 3 months, but it
    # is a real (small) compromise, not a free lunch -- so we list it under Known Limitations.
    beta_df = beta_df.bfill()

    # --- 2) Size proxy = log(price * average volume) ---
    # We want "how big is this company", but true market cap needs point-in-time shares
    # outstanding, which we do not have (and back-filling today's share count into 2018 would
    # be look-ahead bias). Dollar volume is a decent, price-only stand-in.
    # Why the log? Dollar volume is violently right-skewed -- the largest names trade orders of
    # magnitude more than the smallest. OLS is sensitive to high-leverage points, so feeding it
    # a raw skewed regressor lets a couple of mega-caps dominate the fit. Taking logs pulls that
    # tail in and makes the regressor roughly symmetric. The +1e-5 is just a guard so that a
    # zero-volume day never sends log() to -infinity.
    roll_vol = volumes.rolling(window=config.SIZE_PROXY_LOOKBACK).mean()
    size_proxy = np.log(prices * roll_vol + 1e-5).loc[returns.index]

    # --- 3) Sector dummies: one 0/1 indicator column per sector ---
    # Note drop_first=True. If we kept a column for EVERY sector, those columns would sum to
    # exactly 1 for every stock -- which is precisely the intercept column we also include in
    # the regression. That makes the design matrix perfectly collinear, X'X singular, and the
    # coefficients unidentified: the textbook "dummy variable trap". Dropping one sector fixes
    # it, and the remaining coefficients simply read as "difference relative to the omitted
    # sector". Which sector gets dropped has no effect on the residuals, and residuals are the
    # only thing we actually use.
    sector_series = pd.Series(config.SECTOR_MAPPING).reindex(tickers).fillna('Unknown')
    sector_dummies = pd.get_dummies(sector_series, drop_first=True).astype(float)

    return {'beta': beta_df, 'size': size_proxy, 'sectors': sector_dummies}


print("data helpers ready: load_and_clean_data(), compute_risk_exposures()")